In [1]:
!pip install -q kaggle timm albumentations tqdm opencv-python-headless seaborn

import os, zipfile, torch
from pathlib import Path

# Kaggle API credentials
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

with open(kaggle_dir / "kaggle.json", "w") as f:
    f.write('{"username":"ahnafnafim","key":"KGAT_501049cf57c12a07edf49389e126d5cc"}')

os.chmod(kaggle_dir / "kaggle.json", 0o600)

DATA_PARENT = Path("/tmp/data")
DATA_PARENT.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "araftahsanpavel/tgif-subset"
ZIP_PATH = DATA_PARENT / "tgif-subset.zip"
DATA_ROOT = DATA_PARENT / "subset"

if DATA_ROOT.exists():
    print(f"✓ Dataset already extracted at: {DATA_ROOT}")
else:
    print("Downloading TGIF subset from Kaggle...")
    !kaggle datasets download -d {DATASET_SLUG} -p {DATA_PARENT} -q

    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_PARENT)

    ZIP_PATH.unlink(missing_ok=True)
    print(f"✓ Extracted dataset to: {DATA_PARENT}")

print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("\nDataset folders:")
!find /tmp/data -maxdepth 3 -type d | sort | head -80


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
Extracting...
✓ Extracted dataset to: /tmp/data

GPU: NVIDIA H100 80GB HBM3

Dataset folders:
/tmp/data
/tmp/data/subset
/tmp/data/subset/testing
/tmp/data/subset/testing/fakes
/tmp/data/subset/testing/images
/tmp/data/subset/testing/masks
/tmp/data/subset/training
/tmp/data/subset/training/fakes
/tmp/data/subset/training/images
/tmp/data/subset/training/masks
/tmp/data/subset/validation
/tmp/data/subset/validation/fakes
/tmp/data/subset/validation/images
/tmp/data/subset/validation/masks


In [1]:
from __future__ import annotations
import os, time, random, json, copy
import numpy as np
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm
import timm

# ── Hyperparameters ──
IMAGE_SIZE          = 384
ABLATION_EPOCHS     = 15
FULL_EPOCHS         = 30
BATCH_SIZE          = 16
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-4
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
LR_MIN              = 1e-6
EARLY_STOP_PATIENCE = 7
BINARY_THRESHOLD    = 0.5
GRAD_CLIP_NORM      = 1.0
MIXED_PRECISION     = True
SEED                = 42
NUM_WORKERS         = 4
PIN_MEMORY          = True
POS_WEIGHT          = 3.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

DATA_ROOT = Path("/tmp/data/subset")
SAVE_DIR = Path("/root/results_full_novelty_30_modal")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("=" * 70)
print("  Enhanced ForensicNet — Cumulative Ablation Study")
print("=" * 70)
print(f"  Device     : {device}")
if torch.cuda.is_available():
    print(f"  GPU        : {torch.cuda.get_device_name(0)}")
print(f"  IMAGE_SIZE : {IMAGE_SIZE}")
print(f"  ABLATION   : {ABLATION_EPOCHS} epochs")
print(f"  FULL MODEL : {FULL_EPOCHS} epochs")
print(f"  PATIENCE   : {EARLY_STOP_PATIENCE}")
print(f"  SAVE_DIR   : {SAVE_DIR}")
print("=" * 70)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Enhanced ForensicNet — Cumulative Ablation Study
  Device     : cuda
  GPU        : NVIDIA H100 80GB HBM3
  IMAGE_SIZE : 384
  ABLATION   : 15 epochs
  FULL MODEL : 30 epochs
  PATIENCE   : 7
  SAVE_DIR   : /root/results_full_novelty_30_modal


In [2]:
def infer_mask_path_from_fake(fake_path: Path, masks_root: Path) -> Optional[Path]:
    """
    Example fake:
      110999_mask_bbox.png_ps_mask.png_sd2_0.png

    Matching mask:
      110999_mask_bbox.png_ps_mask.png

    This version does NOT restrict to SD2 only.
    """
    category = fake_path.parent.name
    name = fake_path.name

    if "_mask_" not in name:
        return None

    stem = name

    # Remove known generator suffixes
    for tag in ["_sd2_", "_sdxl_", "_firefly_", "_dalle_", "_dalle2_"]:
        if tag in stem:
            stem = stem.split(tag)[0] + ".png"
            break

    candidate = masks_root / category / stem

    if candidate.exists():
        return candidate

    # Fallback: prefix match against available masks
    mask_dir = masks_root / category
    if mask_dir.exists():
        for m in mask_dir.glob("*.png"):
            if name.startswith(m.name):
                return m

    return None


def build_index(split_dir: Path) -> List[Dict[str, Any]]:
    images_root = split_dir / "images"
    masks_root  = split_dir / "masks"
    fakes_root  = split_dir / "fakes"

    samples = []

    # Authentic/original images
    if images_root.exists():
        for p in images_root.rglob("*_orig.png"):
            samples.append({
                "image_path": str(p),
                "mask_path": None,
                "is_fake": False,
                "category": p.parent.name
            })

    # Fake images: include all generators, not only SD2
    if fakes_root.exists():
        for p in fakes_root.rglob("*.png"):
            if "_mask_" not in p.name:
                continue

            paired_mask = infer_mask_path_from_fake(p, masks_root)

            if paired_mask is None or not paired_mask.exists():
                continue

            samples.append({
                "image_path": str(p),
                "mask_path": str(paired_mask),
                "is_fake": True,
                "category": p.parent.name
            })

    return samples


class TGIFDataset(Dataset):
    def __init__(self, samples, image_size):
        self.samples = samples

        # IMPORTANT: no augmentation in ablation.
        self.transform = A.Compose([
            A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        img = cv2.imread(s["image_path"], cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(s["image_path"])

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if s["mask_path"] is not None:
            mask = cv2.imread(s["mask_path"], cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise FileNotFoundError(s["mask_path"])
            mask = (mask >= 127).astype(np.uint8)
        else:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        out = self.transform(image=img, mask=mask)

        return {
            "image": out["image"].float(),
            "mask": out["mask"].float().unsqueeze(0),
            "is_fake": torch.tensor(s["is_fake"], dtype=torch.bool),
        }


print("Indexing dataset ...")
train_samples = build_index(DATA_ROOT / "training")
val_samples   = build_index(DATA_ROOT / "validation")
test_samples  = build_index(DATA_ROOT / "testing")

for name, samples in [("training", train_samples), ("validation", val_samples), ("testing", test_samples)]:
    n_fake = sum(1 for s in samples if s["is_fake"])
    print(f"  {name:<10}: {len(samples):>5} total  ({len(samples)-n_fake:>4} real, {n_fake:>4} fake)")

train_loader = DataLoader(
    TGIFDataset(train_samples, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False
)

val_loader = DataLoader(
    TGIFDataset(val_samples, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    TGIFDataset(test_samples, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"  Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")

Indexing dataset ...
  training  :  7559 total  (2100 real, 5459 fake)
  validation:  1023 total  ( 341 real,  682 fake)
  testing   :  1029 total  ( 343 real,  686 fake)
  Train: 473 batches | Val: 64 | Test: 65


In [3]:
# Cell 4 — Metrics
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)
        inter = (probs * targets).sum(dim=1)
        denom = probs.sum(dim=1) + targets.sum(dim=1)
        return 1.0 - ((2.0 * inter + self.smooth) / (denom + self.smooth)).mean()

class MetricAccumulator:
    def __init__(self): self.tp = self.fp = self.tn = self.fn = 0
    def update(self, preds_binary, targets):
        p = preds_binary.bool(); t = targets.bool()
        self.tp += int((p & t).sum().item())
        self.fp += int((p & ~t).sum().item())
        self.tn += int((~p & ~t).sum().item())
        self.fn += int((~p & t).sum().item())
    def compute(self):
        eps = 1e-8; tp, fp, tn, fn = self.tp, self.fp, self.tn, self.fn
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        return {'pixel_accuracy': (tp + tn) / (tp + fp + tn + fn + eps), 'precision': precision, 'recall': recall, 'f1': f1, 'iou': tp / (tp + fp + fn + eps), 'dice': 2 * tp / (2 * tp + fp + fn + eps), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}
print('✓ Metric helpers ready')


✓ Metric helpers ready


In [13]:
# Cell 5 — Enhanced ForensicNet model
class SRMConv(nn.Module):
    def __init__(self):
        super().__init__()
        f1 = np.array([[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]], dtype=np.float32) / 4.0
        f2 = np.array([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]], dtype=np.float32) / 12.0
        f3 = np.array([[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=np.float32) / 2.0
        self.register_buffer('weight', torch.from_numpy(np.stack([f1, f2, f3])[:, None]))
    def forward(self, x):
        return torch.cat([F.conv2d(x[:, c:c+1], self.weight, padding=2) for c in range(3)], dim=1)

class BayarConv2d(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, k=5):
        super().__init__(); self.k = k; self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.01)
    def forward(self, x):
        w = self.weight.clone(); c = self.k // 2; wn = w.clone(); wn[:, :, c, c] = 0.0; w[:, :, c, c] = -wn.sum(dim=[2,3]); return F.conv2d(x, w, padding=c)

class CBAM(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__(); mid = max(ch // r, 8)
        self.ca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(ch, mid), nn.ReLU(), nn.Linear(mid, ch), nn.Sigmoid())
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = x * self.ca(x).view(x.size(0), -1, 1, 1)
        return x * self.sa(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1))

class FPNBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__(); self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(), nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU())
        self.cbam = CBAM(out_ch)
    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]: x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))

class NoiseBranch(nn.Module):
    def __init__(self, out_ch=128):
        super().__init__(); self.srm = SRMConv(); self.bayar = BayarConv2d(3, 3)
        self.enc = nn.Sequential(nn.Conv2d(12, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(), nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(), nn.Conv2d(64, out_ch, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU())
    def forward(self, x): return self.enc(torch.cat([self.srm(x), self.bayar(x)], dim=1))

class DCTStream(nn.Module):
    """Frequency-inspired 8x8 block stream, not fixed mathematical DCT."""
    def __init__(self, out_channels=128):
        super().__init__()
        self.freq_conv = nn.Sequential(nn.Conv2d(3, 32, kernel_size=8, stride=8, padding=0, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Conv2d(64, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))
    def forward(self, x): return self.freq_conv(x)

class SEFusionBlock(nn.Module):
    def __init__(self, in_ch, out_ch, reduction=16):
        super().__init__(); self.proj = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.se = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(out_ch, max(out_ch // reduction, 4), bias=False), nn.ReLU(inplace=True), nn.Linear(max(out_ch // reduction, 4), out_ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        x = self.proj(x); return x * self.se(x).view(x.shape[0], x.shape[1], 1, 1)

class ContrastiveHead(nn.Module):
    def __init__(self, in_ch, proj_dim=128):
        super().__init__(); self.proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, in_ch, bias=False), nn.ReLU(inplace=True), nn.Linear(in_ch, proj_dim, bias=False))
    def forward(self, x): return F.normalize(self.proj(x), dim=1)

class EnhancedForensicNet(nn.Module):
    def __init__(self, config, image_size=IMAGE_SIZE, pretrained=True):
        super().__init__(); self.image_size = image_size; self.config = config
        self.enc = timm.create_model('convnext_large', pretrained=pretrained, features_only=True, out_indices=(0,1,2,3))
        self.noise = NoiseBranch(out_ch=128)
        if config['use_dct']:
            self.dct = DCTStream(out_channels=128); fusion_in_ch = 1536 + 128 + 128
        else:
            self.dct = None; fusion_in_ch = 1536 + 128
        self.lat4 = SEFusionBlock(fusion_in_ch, 256) if config['use_se_fusion'] else nn.Conv2d(fusion_in_ch, 256, 1)
        self.lat3 = nn.Conv2d(768, 256, 1); self.lat2 = nn.Conv2d(384, 128, 1); self.lat1 = nn.Conv2d(192, 64, 1)
        self.dec3 = FPNBlock(256, 256, 256); self.dec2 = FPNBlock(256, 128, 128); self.dec1 = FPNBlock(128, 64, 64); self.dec0 = FPNBlock(64, 0, 32)
        self.head = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 1, 1))
        self.edge_head = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True), nn.Conv2d(32, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.GELU(), nn.Conv2d(16, 1, 1)) if config['use_edge'] else None
        if config['use_deep_sup']:
            self.deep_head3 = nn.Conv2d(256, 1, 1); self.deep_head2 = nn.Conv2d(128, 1, 1); self.deep_head1 = nn.Conv2d(64, 1, 1)
        else:
            self.deep_head3 = self.deep_head2 = self.deep_head1 = None
        self.contrast_head = ContrastiveHead(256, 128) if config['use_contrastive'] else None
    def forward(self, x):
        s1, s2, s3, s4 = self.enc(x)
        n_feat = F.adaptive_avg_pool2d(self.noise(x), s4.shape[2:])
        if self.dct is not None:
            dct_feat = F.adaptive_avg_pool2d(self.dct(x), s4.shape[2:]); fused_input = torch.cat([s4, n_feat, dct_feat], dim=1)
        else:
            fused_input = torch.cat([s4, n_feat], dim=1)
        p4 = self.lat4(fused_input)
        d3 = self.dec3(p4, self.lat3(s3)); d2 = self.dec2(d3, self.lat2(s2)); d1 = self.dec1(d2, self.lat1(s1)); d0 = self.dec0(d1)
        seg_out = self.head(d0)
        if seg_out.shape[2:] != (self.image_size, self.image_size): seg_out = F.interpolate(seg_out, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
        output = {'mask': seg_out}
        if self.edge_head is not None:
            edge_out = self.edge_head(d0)
            if edge_out.shape[2:] != (self.image_size, self.image_size): edge_out = F.interpolate(edge_out, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
            output['edge'] = edge_out
        if self.deep_head3 is not None: output['deep'] = [self.deep_head3(d3), self.deep_head2(d2), self.deep_head1(d1)]
        if self.contrast_head is not None: output['embed'] = self.contrast_head(p4)
        return output

# Quick verification
_test_config = {'use_dct': True, 'use_se_fusion': True, 'use_edge': True, 'use_deep_sup': True, 'use_contrastive': True}
with torch.no_grad():
    _m = EnhancedForensicNet(_test_config, pretrained=False)
    _o = _m(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE))
    print('✓ Full novelty forward pass verified')
    for k, v in _o.items(): print(k, [x.shape for x in v] if isinstance(v, list) else v.shape)
    print(f'Params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M')
    del _m, _o


✓ Full novelty forward pass verified
mask torch.Size([2, 1, 384, 384])
edge torch.Size([2, 1, 384, 384])
deep [torch.Size([2, 1, 24, 24]), torch.Size([2, 1, 48, 48]), torch.Size([2, 1, 96, 96])]
embed torch.Size([2, 128])
Params: 199.7M


In [8]:
# Cell 6 — Enhanced loss
def _sobel_edges(mask):
    sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=mask.device).view(1,1,3,3)
    sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32, device=mask.device).view(1,1,3,3)
    ex = F.conv2d(mask.float(), sobel_x, padding=1); ey = F.conv2d(mask.float(), sobel_y, padding=1)
    return (torch.sqrt(ex**2 + ey**2 + 1e-8) > 0.5).float()

class EnhancedLoss(nn.Module):
    def __init__(self, config, pos_weight=POS_WEIGHT):
        super().__init__(); self.config = config; self.register_buffer('pw', torch.tensor([pos_weight], dtype=torch.float32)); self.dice = DiceLoss()
        self.w_seg = 1.0; self.w_edge = 0.3 if config['use_edge'] else 0.0; self.w_deep = 0.2 if config['use_deep_sup'] else 0.0; self.w_contrastive = 0.05 if config['use_contrastive'] else 0.0; self.temperature = 0.07
    def _seg_loss(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pw.to(logits.device)); return 0.5*bce + 0.5*self.dice(logits, targets)
    def _edge_loss(self, edge_logits, mask_targets):
        edge_gt = _sobel_edges(mask_targets)
        if edge_logits.shape[-2:] != edge_gt.shape[-2:]: edge_logits = F.interpolate(edge_logits, size=edge_gt.shape[-2:], mode='bilinear', align_corners=True)
        return F.binary_cross_entropy_with_logits(edge_logits, edge_gt)
    def _deep_loss(self, deep_preds, mask_targets):
        total = 0.0
        for pred, w in zip(deep_preds, [0.125, 0.25, 0.5]):
            target_ds = F.interpolate(mask_targets, size=pred.shape[-2:], mode='nearest'); total = total + w*self._seg_loss(pred, target_ds)
        return total
    def _contrastive_loss(self, embeddings, labels):
        labels = labels.view(-1).float(); B = embeddings.shape[0]
        if B < 2: return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature; sim = sim - sim.max(dim=1, keepdim=True).values.detach()
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1); self_mask = ~torch.eye(B, dtype=torch.bool, device=embeddings.device); pos_mask = labels_eq & self_mask
        pos_counts = pos_mask.float().sum(dim=1); valid = pos_counts > 0
        if valid.sum() == 0: return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        exp_sim = torch.exp(sim) * self_mask.float(); log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        loss = -(log_prob * pos_mask.float()).sum(dim=1) / (pos_counts + 1e-8)
        return loss[valid].mean()
    def forward(self, outputs, masks, labels=None):
        l_seg = self._seg_loss(outputs['mask'], masks); total = self.w_seg*l_seg; losses = {'seg': l_seg.detach()}
        if self.config['use_edge'] and 'edge' in outputs:
            l_edge = self._edge_loss(outputs['edge'], masks); total = total + self.w_edge*l_edge; losses['edge'] = l_edge.detach()
        if self.config['use_deep_sup'] and 'deep' in outputs:
            l_deep = self._deep_loss(outputs['deep'], masks); total = total + self.w_deep*l_deep; losses['deep'] = l_deep.detach()
        if self.config['use_contrastive'] and 'embed' in outputs and labels is not None:
            l_con = self._contrastive_loss(outputs['embed'], labels); total = total + self.w_contrastive*l_con; losses['contrastive'] = l_con.detach()
        losses['total'] = total; return losses
print('✓ EnhancedLoss ready')


✓ EnhancedLoss ready


In [9]:
# Cell 7 — Training engine
def run_one_epoch(model, loader, criterion, optimizer, scaler, device, train, desc):
    model.train(mode=train); total_loss = 0.0; n_batches = 0; metrics = MetricAccumulator(); grad_ctx = torch.enable_grad() if train else torch.no_grad()
    with grad_ctx:
        pbar = tqdm(loader, desc=desc, leave=False)
        for batch in pbar:
            images = batch['image'].to(device, non_blocking=True); targets = batch['mask'].to(device, non_blocking=True); labels = batch['is_fake'].to(device, non_blocking=True)
            if train: optimizer.zero_grad(set_to_none=True)
            amp_on = MIXED_PRECISION and device.type == 'cuda'
            with autocast(device_type='cuda', enabled=amp_on):
                outputs = model(images); loss_dict = criterion(outputs, targets, labels); loss = loss_dict['total']
            if train:
                if scaler and amp_on:
                    scaler.scale(loss).backward(); scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM); optimizer.step()
            with torch.no_grad(): metrics.update((torch.sigmoid(outputs['mask']) >= BINARY_THRESHOLD).float(), targets)
            total_loss += loss.item(); n_batches += 1; pbar.set_postfix(loss=f'{loss.item():.4f}')
    return total_loss / max(n_batches, 1), metrics.compute()


def train_config(config_name, config, epochs, keep_checkpoint=False):
    print(f"\n{'='*70}\nCONFIG: {config_name}")
    active = [k.replace('use_', '') for k, v in config.items() if v]
    print(f"Active novelty: {active if active else ['none']} | Epochs: {epochs}\n{'='*70}")
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = EnhancedForensicNet(config, pretrained=True).to(device)
    total_params = sum(p.numel() for p in model.parameters()); print(f'Params: {total_params/1e6:.1f}M | Size: {total_params*4/1024**2:.0f}MB')
    criterion = EnhancedLoss(config, pos_weight=POS_WEIGHT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN)
    scaler = GradScaler(enabled=MIXED_PRECISION and device.type == 'cuda')
    best_val_iou = -1.0; best_epoch = 0; epochs_no_improve = 0; history = []
    ckpt_path = SAVE_DIR / f'best_{config_name}.pth'
    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        train_loss, train_metrics = run_one_epoch(model, train_loader, criterion, optimizer, scaler, device, train=True, desc=f'{config_name} train {epoch}')
        val_loss, val_metrics = run_one_epoch(model, val_loader, criterion, None, None, device, train=False, desc=f'{config_name} val {epoch}')
        current_lr = optimizer.param_groups[0]['lr']; epoch_secs = time.time() - epoch_start; scheduler.step(val_metrics['iou'])
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'train_iou': train_metrics['iou'], 'val_iou': val_metrics['iou'], 'train_f1': train_metrics['f1'], 'val_f1': val_metrics['f1'], 'val_precision': val_metrics['precision'], 'val_recall': val_metrics['recall'], 'lr': current_lr, 'epoch_seconds': epoch_secs})
        if val_metrics['iou'] > best_val_iou:
            best_val_iou = val_metrics['iou']; best_epoch = epoch; epochs_no_improve = 0
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'best_val_iou': best_val_iou, 'config': config, 'config_name': config_name, 'history': history}, ckpt_path)
            print(f"Epoch {epoch:>3}/{epochs} | val IoU {val_metrics['iou']:.4f} | val F1 {val_metrics['f1']:.4f} | lr {current_lr:.2e} | {epoch_secs:.0f}s ✓ BEST")
        else:
            epochs_no_improve += 1
            if epoch % 5 == 0 or epochs_no_improve >= EARLY_STOP_PATIENCE - 1:
                print(f"Epoch {epoch:>3}/{epochs} | val IoU {val_metrics['iou']:.4f} | val F1 {val_metrics['f1']:.4f} | lr {current_lr:.2e} | {epoch_secs:.0f}s | patience {epochs_no_improve}/{EARLY_STOP_PATIENCE}")
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f'[early stop] at epoch {epoch}'); break
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False); model.load_state_dict(ckpt['model'])
    test_loss, test_metrics = run_one_epoch(model, test_loader, criterion, None, None, device, train=False, desc=f'{config_name} test')
    print(f"\nTEST — IoU: {test_metrics['iou']:.4f} | F1: {test_metrics['f1']:.4f} | Prec: {test_metrics['precision']:.4f} | Rec: {test_metrics['recall']:.4f}")
    result = {'config_name': config_name, 'config': config, 'epochs_trained': history[-1]['epoch'] if history else 0, 'best_epoch': best_epoch, 'best_val_iou': best_val_iou, 'test_metrics': test_metrics, 'history': history, 'total_params': total_params, 'model_size_mb': total_params * 4 / 1024**2, 'checkpoint_kept': bool(keep_checkpoint and ckpt_path.exists()), 'checkpoint_path': str(ckpt_path) if keep_checkpoint and ckpt_path.exists() else None}
    with open(SAVE_DIR / f'{config_name}_results.json', 'w') as f: json.dump(result, f, indent=2, default=str)
    if ckpt_path.exists() and not keep_checkpoint: os.remove(ckpt_path)
    elif ckpt_path.exists() and keep_checkpoint: print(f'Kept checkpoint: {ckpt_path}')
    del model, criterion, optimizer, scheduler, scaler; torch.cuda.empty_cache()
    return result
print('✓ Training engine ready')


✓ Training engine ready


In [11]:
# ============================================================
# Cell 7 — Training engine
# ============================================================

def run_one_epoch(model, loader, criterion, optimizer, scaler, device, train, desc):
    model.train(mode=train)

    total_loss, n_batches = 0.0, 0
    metrics = MetricAccumulator()
    grad_ctx = torch.enable_grad() if train else torch.no_grad()

    with grad_ctx:
        pbar = tqdm(loader, desc=desc, leave=False)

        for batch in pbar:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["mask"].to(device, non_blocking=True)
            labels = batch["is_fake"].to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            amp_on = MIXED_PRECISION and device.type == "cuda"

            with autocast(device_type="cuda", enabled=amp_on):
                outputs = model(images)
                loss_dict = criterion(outputs, targets, labels)
                loss = loss_dict["total"]

            if train:
                if scaler and amp_on:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    optimizer.step()

            with torch.no_grad():
                preds = (torch.sigmoid(outputs["mask"]) >= BINARY_THRESHOLD).float()
                metrics.update(preds, targets)

            total_loss += loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(n_batches, 1), metrics.compute()


def train_config(config_name, config, epochs, keep_checkpoint=False):
    print(f"\n{'='*70}")
    print(f"CONFIG: {config_name}")
    active = [k.replace("use_", "") for k, v in config.items() if v]
    print(f"Active novelty: {active if active else ['none']} | Epochs: {epochs}")
    print(f"{'='*70}")

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = EnhancedForensicNet(config, pretrained=True).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    model_size_mb = total_params * 4 / 1024**2
    print(f"Params: {total_params/1e6:.1f}M | Size: {model_size_mb:.0f}MB")

    criterion = EnhancedLoss(config, pos_weight=POS_WEIGHT).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=LR_MIN
    )

    scaler = GradScaler(enabled=MIXED_PRECISION and device.type == "cuda")

    best_val_iou = -1.0
    best_epoch = 0
    epochs_no_improve = 0
    history = []

    ckpt_path = SAVE_DIR / f"best_{config_name}.pth"

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()

        train_loss, train_metrics = run_one_epoch(
            model, train_loader, criterion, optimizer, scaler,
            device, train=True, desc=f"{config_name} train {epoch}"
        )

        val_loss, val_metrics = run_one_epoch(
            model, val_loader, criterion, None, None,
            device, train=False, desc=f"{config_name} val {epoch}"
        )

        current_lr = optimizer.param_groups[0]["lr"]
        epoch_secs = time.time() - epoch_start
        scheduler.step(val_metrics["iou"])

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_iou": train_metrics["iou"],
            "val_iou": val_metrics["iou"],
            "train_f1": train_metrics["f1"],
            "val_f1": val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "lr": current_lr,
            "epoch_seconds": epoch_secs,
        })

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            best_epoch = epoch
            epochs_no_improve = 0

            torch.save({
                "model": model.state_dict(),
                "epoch": epoch,
                "best_val_iou": best_val_iou,
                "best_val_f1": val_metrics["f1"],
                "config": config,
                "config_name": config_name,
                "history": history,
                "total_params": total_params,
                "model_size_mb": model_size_mb,
            }, ckpt_path)

            print(f"Epoch {epoch:>3}/{epochs} | "
                  f"val IoU {val_metrics['iou']:.4f} | "
                  f"val F1 {val_metrics['f1']:.4f} | "
                  f"lr {current_lr:.2e} | "
                  f"{epoch_secs:.0f}s ✓ BEST")

        else:
            epochs_no_improve += 1

            if epoch % 5 == 0 or epochs_no_improve >= EARLY_STOP_PATIENCE - 1:
                print(f"Epoch {epoch:>3}/{epochs} | "
                      f"val IoU {val_metrics['iou']:.4f} | "
                      f"val F1 {val_metrics['f1']:.4f} | "
                      f"lr {current_lr:.2e} | "
                      f"{epoch_secs:.0f}s | "
                      f"patience {epochs_no_improve}/{EARLY_STOP_PATIENCE}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"[early stop] at epoch {epoch}")
            break

    # Load best checkpoint for final test evaluation
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model"])

    test_loss, test_metrics = run_one_epoch(
        model, test_loader, criterion, None, None,
        device, train=False, desc=f"{config_name} test"
    )

    print(f"\nTEST — IoU: {test_metrics['iou']:.4f} | "
          f"F1: {test_metrics['f1']:.4f} | "
          f"Prec: {test_metrics['precision']:.4f} | "
          f"Rec: {test_metrics['recall']:.4f}")

    result = {
        "config_name": config_name,
        "config": config,
        "epochs_trained": history[-1]["epoch"] if history else 0,
        "best_epoch": best_epoch,
        "best_val_iou": best_val_iou,
        "test_metrics": test_metrics,
        "history": history,
        "total_params": total_params,
        "model_size_mb": model_size_mb,
        "checkpoint_kept": bool(keep_checkpoint and ckpt_path.exists()),
        "checkpoint_path": str(ckpt_path) if keep_checkpoint and ckpt_path.exists() else None,
    }

    with open(SAVE_DIR / f"{config_name}_results.json", "w") as f:
        json.dump(result, f, indent=2, default=str)

    if ckpt_path.exists() and not keep_checkpoint:
        os.remove(ckpt_path)
    elif ckpt_path.exists() and keep_checkpoint:
        print(f"Kept checkpoint: {ckpt_path}")

    del model, criterion, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

    return result


print("✓ train_config is now defined.")

✓ train_config is now defined.


In [14]:
# ============================================================
# Cell 8 — FULL NOVELTY ONLY, 30 EPOCHS — Modal version
# ============================================================

from pathlib import Path
import json, os, zipfile
import matplotlib.pyplot as plt

# Modal-safe output directory
# Download/zip this before closing the Modal session.
SAVE_DIR = Path("/root/results_full_novelty_30_modal")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Train full novelty for up to 30 epochs
FULL_EPOCHS = 30

# Recommended for Modal credits: allow early stopping
# Use 999 only if you truly want to force all 30 epochs.
EARLY_STOP_PATIENCE = 7

print("=" * 70)
print("FULL NOVELTY TRAINING ONLY — 30 EPOCHS — MODAL")
print("=" * 70)
print(f"SAVE_DIR             : {SAVE_DIR}")
print(f"FULL_EPOCHS          : {FULL_EPOCHS}")
print(f"EARLY_STOP_PATIENCE  : {EARLY_STOP_PATIENCE}")
print(f"Device               : {device}")
if torch.cuda.is_available():
    print(f"GPU                  : {torch.cuda.get_device_name(0)}")
print("=" * 70)


BASELINE_FORENSICNET = {
    "config_name": "baseline_forensicnet_reference",
    "config": {
        "use_dct": False,
        "use_se_fusion": False,
        "use_edge": False,
        "use_deep_sup": False,
        "use_contrastive": False
    },
    "test_metrics": {
        "pixel_accuracy": 0.9931,
        "precision": 0.9252,
        "recall": 0.9108,
        "f1": 0.9179,
        "iou": 0.8483,
        "dice": 0.9179
    },
    "best_val_iou": 0.8441,
    "best_val_f1": 0.9155,
    "best_epoch": 18,
    "epochs_trained": 25,
    "total_time_seconds": 15138,
    "avg_epoch_seconds": 586,
    "total_params": 199560000,
    "model_size_mb": 761.3,
    "note": (
        "Fixed reference result from already-trained ForensicNet. "
        "Best validation IoU was 0.8441 at epoch 18; final test IoU was 0.8483."
    )
}


all_results = {
    "baseline_forensicnet": BASELINE_FORENSICNET,

    "cumulative_1_dct": {
        "config_name": "cumulative_1_dct",
        "config": {
            "use_dct": True,
            "use_se_fusion": False,
            "use_edge": False,
            "use_deep_sup": False,
            "use_contrastive": False
        },
        "test_metrics": {
            "pixel_accuracy": None,
            "precision": 0.8949,
            "recall": 0.9051,
            "f1": 0.8999,
            "iou": 0.8181,
            "dice": 0.8999
        },
        "note": "Reconstructed from Modal printed output."
    },

    "cumulative_2_dct_se": {
        "config_name": "cumulative_2_dct_se",
        "config": {
            "use_dct": True,
            "use_se_fusion": True,
            "use_edge": False,
            "use_deep_sup": False,
            "use_contrastive": False
        },
        "test_metrics": {
            "pixel_accuracy": None,
            "precision": 0.9250,
            "recall": 0.8984,
            "f1": 0.9115,
            "iou": 0.8374,
            "dice": 0.9115
        },
        "note": "Reconstructed from Modal printed output."
    },

    "cumulative_3_dct_se_edge": {
        "config_name": "cumulative_3_dct_se_edge",
        "config": {
            "use_dct": True,
            "use_se_fusion": True,
            "use_edge": True,
            "use_deep_sup": False,
            "use_contrastive": False
        },
        "test_metrics": {
            "pixel_accuracy": None,
            "precision": 0.9096,
            "recall": 0.9061,
            "f1": 0.9079,
            "iou": 0.8313,
            "dice": 0.9079
        },
        "note": "Reconstructed from Modal printed output."
    },

    "cumulative_4_dct_se_edge_deep": {
        "config_name": "cumulative_4_dct_se_edge_deep",
        "config": {
            "use_dct": True,
            "use_se_fusion": True,
            "use_edge": True,
            "use_deep_sup": True,
            "use_contrastive": False
        },
        "test_metrics": {
            "pixel_accuracy": None,
            "precision": 0.9112,
            "recall": 0.9044,
            "f1": 0.9078,
            "iou": 0.8312,
            "dice": 0.9078
        },
        "note": "Reconstructed from Modal printed output."
    }
}


FULL_NOVELTY_CONFIG = {
    "use_dct": True,
    "use_se_fusion": True,
    "use_edge": True,
    "use_deep_sup": True,
    "use_contrastive": True
}


result = train_config(
    config_name="full_novelty_30epochs",
    config=FULL_NOVELTY_CONFIG,
    epochs=FULL_EPOCHS,
    keep_checkpoint=True
)

all_results["full_novelty"] = result


partial_json = SAVE_DIR / "full_novelty_30_with_reconstructed_cumulative.json"

with open(partial_json, "w") as f:
    json.dump(all_results, f, indent=2, default=str)

print("\n✓ Full novelty training complete.")
print(f"✓ Saved combined results: {partial_json}")

FULL NOVELTY TRAINING ONLY — 30 EPOCHS — MODAL
SAVE_DIR             : /root/results_full_novelty_30_modal
FULL_EPOCHS          : 30
EARLY_STOP_PATIENCE  : 7
Device               : cuda
GPU                  : NVIDIA H100 80GB HBM3

CONFIG: full_novelty_30epochs
Active novelty: ['dct', 'se_fusion', 'edge', 'deep_sup', 'contrastive'] | Epochs: 30


Params: 199.7M | Size: 762MB


Epoch   1/30 | val IoU 0.0950 | val F1 0.1734 | lr 1.00e-04 | 120s ✓ BEST


Epoch   2/30 | val IoU 0.4731 | val F1 0.6424 | lr 1.00e-04 | 109s ✓ BEST


Epoch   3/30 | val IoU 0.6231 | val F1 0.7678 | lr 1.00e-04 | 109s ✓ BEST


...

Epoch  20/30 | val IoU 0.8317 | val F1 0.9081 | lr 5.00e-05 | 108s ✓ BEST


Epoch  25/30 | val IoU 0.8223 | val F1 0.9025 | lr 2.50e-05 | 106s | patience 5/7


Epoch  26/30 | val IoU 0.8100 | val F1 0.8951 | lr 2.50e-05 | 105s | patience 6/7


Epoch  27/30 | val IoU 0.8146 | val F1 0.8978 | lr 2.50e-05 | 107s | patience 7/7
[early stop] at epoch 27



TEST — IoU: 0.8429 | F1: 0.9147 | Prec: 0.9162 | Rec: 0.9132
Kept checkpoint: /root/results_full_novelty_30_modal/best_full_novelty_30epochs.pth

✓ Full novelty training complete.
✓ Saved combined results: /root/results_full_novelty_30_modal/full_novelty_30_with_reconstructed_cumulative.json


In [15]:
!find /root -name "*.pth" -type f 2>/dev/null
!find / -name "best_full_novelty_30epochs.pth" -type f 2>/dev/null

/root/results_full_novelty_30_modal/best_full_novelty_30epochs.pth
/root/results_full_novelty_30_modal/best_full_novelty_30epochs.pth


In [16]:
!ls -lh /root/results_full_novelty_30_modal

total 763M
-rw-r--r-- 1 root root 763M Apr 29 19:37 best_full_novelty_30epochs.pth
-rw-r--r-- 1 root root  15K Apr 29 19:49 full_novelty_30_with_reconstructed_cumulative.json
-rw-r--r-- 1 root root  12K Apr 29 19:49 full_novelty_30epochs_results.json


In [17]:
import json
from pathlib import Path

SAVE_DIR = Path("/root/results_full_novelty_30_modal")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

final_metrics = {
    "model": "full_novelty_30epochs",
    "config": {
        "use_dct": True,
        "use_se_fusion": True,
        "use_edge": True,
        "use_deep_sup": True,
        "use_contrastive": True,
        "epochs_requested": 30,
        "early_stopped_at": 27,
        "best_epoch": 20,
        "image_size": 384,
        "batch_size": 16,
        "pos_weight": 3.0,
        "augmentation": False
    },
    "validation": {
        "best_val_iou": 0.8317,
        "best_val_f1": 0.9081
    },
    "test": {
        "iou": 0.8429,
        "f1": 0.9147,
        "precision": 0.9162,
        "recall": 0.9132
    },
    "reference_baseline_forensicnet": {
        "test_iou": 0.8483,
        "test_f1": 0.9179,
        "precision": 0.9252,
        "recall": 0.9108,
        "best_val_iou": 0.8441,
        "best_val_f1": 0.9155
    },
    "cumulative_ablation_results": {
        "dct_only": {
            "iou": 0.8181,
            "f1": 0.8999,
            "precision": 0.8949,
            "recall": 0.9051
        },
        "dct_se": {
            "iou": 0.8374,
            "f1": 0.9115,
            "precision": 0.9250,
            "recall": 0.8984
        },
        "dct_se_edge": {
            "iou": 0.8313,
            "f1": 0.9079,
            "precision": 0.9096,
            "recall": 0.9061
        },
        "dct_se_edge_deep": {
            "iou": 0.8312,
            "f1": 0.9078,
            "precision": 0.9112,
            "recall": 0.9044
        },
        "full_novelty": {
            "iou": 0.8429,
            "f1": 0.9147,
            "precision": 0.9162,
            "recall": 0.9132
        }
    }
}

json_path = SAVE_DIR / "full_novelty_30epochs_manual_results.json"

with open(json_path, "w") as f:
    json.dump(final_metrics, f, indent=2)

print(f"✓ Saved manual results JSON: {json_path}")

✓ Saved manual results JSON: /root/results_full_novelty_30_modal/full_novelty_30epochs_manual_results.json
